# 🫁 LungRay: ResNet-50 Consensus Backbone Multi-Task Training
## Kaggle GPU 1-Click Execution Notebook

This notebook trains the **ResNet-50** candidate backbone to complete the **Triple-Backbone Consensus Ensemble**:
- **Shared Encoder**: ResNet-50 (23.5M parameters, deep residual gradient highways).
- **Multi-Task Heads**: Parameter-free LayerNorm feature projection for Head A (14 NIH Pathologies), Head B (TB Screening), Head C (COVIDx CXR-4).
- **Loss Formulation**: Asymmetric Loss (ASL: gamma_neg=4, gamma_pos=1, m=0.05) + Balanced Multi-Task Masked Loss.
- **Sampling**: `WeightedMultiTaskBatchSampler` maintaining 60% NIH : 25% COVIDx : 15% TB equilibrium.
- **Output**: `/kaggle/working/best_resnet50.pth`.

In [ ]:
# 1. Environment & GPU Verification
import os
import sys
import torch

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"CUDA Accelerator: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 * 1024 * 1024)
    print(f"VRAM Available: {vram_gb:.2f} GB")
else:
    print("[!] Warning: CUDA is not active. Please enable GPU accelerator in Kaggle Settings.")

In [ ]:
# 2. Clone / Setup LungRay Repository Code
!git clone https://github.com/philipsgn/LungRay.git /kaggle/working/LungRay
sys.path.insert(0, "/kaggle/working/LungRay")
os.chdir("/kaggle/working/LungRay")
print("[OK] LungRay workspace ready at:", os.getcwd())

In [ ]:
# 3. Launch ResNet-50 Multi-Task Training
!python kaggle/train_resnet50_kaggle.py \
    --epochs 15 \
    --batch-size 16 \
    --lr 1e-4 \
    --weight-decay 1e-4 \
    --backbone resnet50 \
    --head-a-criterion asl \
    --asl-gamma-neg 4.0 \
    --asl-gamma-pos 1.0 \
    --asl-clip 0.05 \
    --max-batches 600 \
    --save-name best_resnet50.pth

In [ ]:
# 4. Verify & Copy Checkpoint to Working Directory
import shutil
from pathlib import Path

ckpt_src = Path("/kaggle/working/LungRay/checkpoints/best_resnet50.pth")
ckpt_dst = Path("/kaggle/working/best_resnet50.pth")

if ckpt_src.exists():
    shutil.copy(ckpt_src, ckpt_dst)
    size_mb = ckpt_dst.stat().st_size / (1024 * 1024)
    print(f"[SUCCESS] Checkpoint saved for download at: {ckpt_dst} ({size_mb:.1f} MB)")
else:
    print("[!] Checkpoint not found in checkpoints directory.")